# First L0 processor example, version==0.9.0

https://pforge-exchange2.astrium.eads.net/jira/browse/RSPY-607

See the associated:

  * Python module: [first_l0_processor.py](./first_l0_processor.py)
  * YAML file: [first_l0_processor.yaml](./first_l0_processor.yaml)

## Initialization

In [ ]:
import os
print(f"Prefect server URL used internally: {os.environ['PREFECT_API_URL']}")
dashboard = f"{os.environ['RSPY_PREFECT_URL']}/dashboard"
print(f"Prefect dashboard public URL: {dashboard}")

In [ ]:
# Init environment before running a demo notebook.
from resources.utils import *  
from resources.dask_utils import *
from resources.prefect_utils import *

init_demo()
init_dask_cluster_eopf(scale=3)
# , image="ghcr.io/rs-python/rs-infrastructure-dask-eopf:feat-rspy607-l0-processing" # temp

# Reload the global vars again
from resources.utils import *  
from resources.dask_utils import *  
from resources.prefect_utils import * 

In [ ]:
# Other imports
import getpass
import os
from importlib import reload
from prefect_aws import S3Bucket
from resources import prefect_utils

# Create a S3 access from the existing credentials from the prefect block
block_s3 = S3Bucket(
    bucket_name="rs-cluster-temp",
    credentials=PREFECT_BLOCK_S3.credentials,
    bucket_folder="",
)

# All our data for this demo is under this bucket
BUCKET_NAME = "rs-cluster-temp"

# Depending on the functions we call, we need to pass a different s3 bucket folder level (yikes)
class MyS3Folder:
    def __init__(self, subdir): # subdir from the bucket name
        self.subdir = subdir
    def subdir():
        return self.subdir
    def prefix_subdir: # 

    

# For each data: 
# input_config_folder: s3 bucket folder that contains the configuration files (NOT THE VOLUMINOUS DATA !).
# It will be downloaded locally.
# payload_file: input yaml configuration file to pass to the triggering. Local to the 'input_config_folder'.
# output_data_folder: s3 bucket folder that will contain the generated data.
short_s1 = {
    "input_config_folder": "s3://rs-cluster-temp/stations/dpr-test/l0/input/s1",
    "payload_file": "iw_joborder.short.yaml",
    "output_data_folder": "s3://rs-cluster-temp/stations/dpr-test/l0/output/s1.short",
}
s1 = {
    "input_config_folder": "s3://rs-cluster-temp/stations/dpr-test/l0/input/s1",
    "payload_file": "iw_joborder.yaml",
    "output_data_folder": "s3://rs-cluster-temp/stations/dpr-test/l0/output/s1",
}
s3 = {
    "input_config_folder": "s3://rs-cluster-temp/stations/dpr-test/l0/input/s3",
    "payload_file": "s3_dordop_payload.yaml",
    "output_data_folder": "s3://rs-cluster-temp/stations/dpr-test/l0/output/s3",
}

# Convert to json to trigger prefect flow
def to_json(my_data):
    return json.dumps(my_data).replace('"', r'\"')

In [ ]:
# We use only the EOPF dask cluster in this tutorial
dask_gateway = dask_gateway_eopf
dask_client = dask_client_eopf
dask_cluster = dask_cluster_eopf
if local_mode:
    os.environ["DASK_GATEWAY_ADDRESS"] = os.environ["DASK_GATEWAY_EOPF_ADDRESS"]

# Save cluster info to be read by our flow
os.environ["DASK_CLUSTER_NAME"] = dask_cluster.name

In [ ]:
# NOTE: we need to create the S3 folder with a dummy file before running DPR
!echo "empty" > "/tmp/.empty"
await PREFECT_BLOCK_S3.upload_from_path("/tmp/.empty", f"{s3_subdir}/.empty")

## Deploy Prefect flow

We deploy our source code via the S3 bucket.

In [ ]:
# Use a subfolder named after the current user
s3_code_folder = f"users/{os.environ.get('RSPY_HOST_USER', getpass.getuser())}/code" 

if local_mode:
    print (f"S3 MinIO dashboard: http://localhost:9101 with user=minio password=Strong#Pass#1234")
print(f"Upload local source code to: 's3://{PREFECT_BLOCK_S3.bucket_name}/{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}'")

# Upload local directory contents
await PREFECT_BLOCK_S3.put_directory(local_path = ".", to_path = s3_code_folder)

# It doesn't follow symlinks so upload them manually
await PREFECT_BLOCK_S3.put_directory(local_path = "./resources", to_path = f"{s3_code_folder}/resources")

# Pass the full S3 code folder as an environment variable
os.environ["S3_CODE_FOLDER"] = f"{PREFECT_BLOCK_S3.bucket_folder}/{s3_code_folder}"

In [ ]:
%%bash
# Deploy the flow. We don't need to be in the git root folder.
prefect --no-prompt deploy --prefect-file "./dpr_processor_example.yaml"

In [ ]:
deploy_name = "dpr-flow/sprint20-dpr-example"
await prefect_utils.wait_for_deployment(deploy_name)

## Run Prefect flow

In [ ]:
print(f"Remove existing zarr products from: {s3_full_path!r}")
PREFECT_BLOCK_S3._get_bucket_resource().objects.filter(Prefix=f"{s3_prefix_subdir}/{s3_basename}").delete()

In [ ]:
%%bash -s "$deploy_name" "$my_data_str"
# Trigger a run for this flow from the command line
prefect deployment run "$1" --params "$2" --watch

NOTE: we could also call the Prefect flow from Python code.
This is useful to debug or see the generated HTML representation.

In [ ]:
run_from_python = False
if run_from_python:
    # Import the module, or reload it if you changed its source code
    import dpr_processor_example
    reload(dpr_processor_example)
    
    # Run the flow
    results = dpr_processor_example.dpr_flow(**my_data)
    
    # Display HTML representation
    import IPython
    for result in results:
        display(IPython.display.HTML(result))
    del results

## Check results

In [ ]:
# Download zarr products into local
local_dir = "/tmp/zarr/"
!rm -rf "$local_dir" && mkdir -p "$local_dir"
await PREFECT_BLOCK_S3.get_directory(f"{s3_prefix_subdir}", local_dir)
!ls -al "$local_dir"

In [ ]:
# Open them with the zarr python package
# see: https://help.marine.copernicus.eu/en/articles/8077952-how-to-open-and-visualize-zarr-format-data
!pip install zarr
import zarr

for filename in s3_filenames:
    store = zarr.open(f"{local_dir}/{filename}.zarr")
    display(store.tree())
    
    # Read some data
    zarr_array = store["measurements"]["image"]["sensor1"]
    display(zarr_array)
    
    # Read the data into memory as a NumPy array
    numpy_array = zarr_array[:]
    display(numpy_array)

In [ ]:


print(f"Output zarr products will be written to: {s3_full_path}")

## 3. Shutdown the dask clusters

In [ ]:
if local_mode:

    # You can scale the clusters to 0 workers
    dask_gateway.scale_cluster(dask_cluster.name, 0)

    # Or shutdown the clusters
    shutdown_dask_clusters(dask_gateway, dask_cluster.name)

# Close the python objects
close_dask_clusters()

# NOTE: restart your python kernel or terminal after the shutdown
# to avoid strange behaviour.